# Retrieval & Reranking Evaluation
This notebook evaluates different retrieval and reranking strategies for plant disease information.

**Comparisons:**
1.  **Embeddings**: Google `gemini-embedding-001` vs OpenAI `text-embedding-3-large`
2.  **Sparse/Hybrid**: `SPLADE` vs `BM25`
3.  **Rerankers**: `Voyage AI rerank-2.5` vs `jina-colbert-v2` (late interaction via Qdrant)

**Metrics:**
- NDCG@5
- Recall@5


In [2]:
import os
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from dotenv import load_dotenv
from pathlib import Path
import uuid

# Vector DB & Embeddings
from qdrant_client import QdrantClient, models
from langchain_qdrant import FastEmbedSparse
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_openai import OpenAIEmbeddings
from fastembed import LateInteractionTextEmbedding

# Reranking
import voyageai

# Evaluation
from ranx import Qrels, Run, evaluate
from rank_bm25 import BM25Okapi

# Load environment variables
load_dotenv()

# Configuration
class Config:
    QDRANT_URL = os.getenv("http://localhost:6333")
    COLLECTION_NAME = "knowledgebase_collection"
    
    # API Keys
    GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
    OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
    OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
    VOYAGE_API_KEY = os.getenv("VOYAGE_API_KEY")
    
    # Models
    GEMINI_MODEL = "models/gemini-embedding-001"
    OPENAI_MODEL = "text-embedding-3-large"
    COLBERT_MODEL = "jinaai/jina-colbert-v2"
    
    # Paths
    CORPUS_PATH = Path("data/corpus.parquet")
    QA_PATH = Path("data/qa.parquet")

# Initialize Clients
client = QdrantClient(url=Config.QDRANT_URL)


In [ ]:
LateInteractionTextEmbedding.list_supported_models()

[{'model': 'colbert-ir/colbertv2.0',
  'sources': {'hf': 'colbert-ir/colbertv2.0',
   'url': None,
   '_deprecated_tar_struct': False},
  'model_file': 'model.onnx',
  'description': 'Late interaction model',
  'license': 'mit',
  'size_in_GB': 0.44,
  'additional_files': [],
  'dim': 128,
  'tasks': {}},
 {'model': 'answerdotai/answerai-colbert-small-v1',
  'sources': {'hf': 'answerdotai/answerai-colbert-small-v1',
   'url': None,
   '_deprecated_tar_struct': False},
  'model_file': 'vespa_colbert.onnx',
  'description': 'Text embeddings, Unimodal (text), Multilingual (~100 languages), 512 input tokens truncation, 2024 year',
  'license': 'apache-2.0',
  'size_in_GB': 0.13,
  'additional_files': [],
  'dim': 96,
  'tasks': {}},
 {'model': 'jinaai/jina-colbert-v2',
  'sources': {'hf': 'jinaai/jina-colbert-v2',
   'url': None,
   '_deprecated_tar_struct': False},
  'model_file': 'onnx/model.onnx',
  'description': 'New model that expands capabilities of colbert-v1 with multilingual and 

In [ ]:
# Load Data
if Config.CORPUS_PATH.exists() and Config.QA_PATH.exists():
    corpus_df = pd.read_parquet(Config.CORPUS_PATH)
    qa_df = pd.read_parquet(Config.QA_PATH)
    print(f"Loaded Corpus: {len(corpus_df)} documents")
    print(f"Loaded QA: {len(qa_df)} queries")
else:
    raise FileNotFoundError("Corpus or QA dataset not found.")

# Prepare Qrels (Ground Truth) for ranx
# qa_df['retrieval_gt'] contains list of list of doc_ids. 
# We need to convert it to a dictionary format for ranx: {qid: {doc_id: score}}
# Assuming binary relevance (1) for ground truth documents.

qrels_dict = {}
for _, row in qa_df.iterrows():
    qid = str(row['qid'])
    # retrieval_gt is often a list of lists in AutoRAG/some formats, or just list of strings.
    # Based on explore.ipynb output: [[17098978-5198-42ce-a808-a6c6b237f383]]
    # It seems to be a list of lists.
    
    gt_ids = row['retrieval_gt']
    qrels_dict[qid] = {}
    
    # Handle potential list of lists or simple list
    # Based on the sanity check, it seems we might have numpy arrays or lists wrapping the IDs
    for item in gt_ids:
        # If item is a list or numpy array, iterate through it
        if isinstance(item, (list, np.ndarray)):
            for doc_id in item:
                qrels_dict[qid][str(doc_id)] = 1
        else:
            # If it's a string or other scalar, use it directly
            qrels_dict[qid][str(item)] = 1

qrels = Qrels(qrels_dict)
print("Qrels prepared.")


Loaded Corpus: 2469 documents
Loaded QA: 600 queries
Qrels prepared.


## Sanity Check: Verify Qrels Setup
Before running the full evaluation, let's verify the qrels are properly formatted and check the ground truth data.

In [ ]:
# Sanity Check 1: Inspect Qrels Structure
print("=== QRELS SANITY CHECK ===")
print(f"Number of queries in qrels: {len(qrels_dict)}")
print(f"\nFirst 3 qrels entries:")
for i, (qid, docs) in enumerate(list(qrels_dict.items())[:3]):
    print(f"\nQuery ID: {qid}")
    print(f"  Ground truth docs: {docs}")
    
# Sanity Check 2: Inspect QA DataFrame
print("\n=== QA DATAFRAME CHECK ===")
print(f"QA DataFrame columns: {qa_df.columns.tolist()}")
print(f"\nFirst query example:")
first_row = qa_df.iloc[0]
print(f"  QID: {first_row['qid']}")
print(f"  Query: {first_row['query']}")
print(f"  Retrieval GT: {first_row['retrieval_gt']}")
print(f"  GT Type: {type(first_row['retrieval_gt'])}")

# Sanity Check 3: Verify corpus doc_ids format
print("\n=== CORPUS DOC_ID CHECK ===")
print(f"First 3 corpus doc_ids:")
for i in range(min(3, len(corpus_df))):
    print(f"  {corpus_df.iloc[i]['doc_id']} (type: {type(corpus_df.iloc[i]['doc_id'])})")

# Sanity Check 4: Check if ground truth IDs exist in corpus
print("\n=== ID MATCH CHECK ===")
corpus_ids_set = set(str(doc_id) for doc_id in corpus_df['doc_id'].tolist())
total_gt_docs = 0
matched_gt_docs = 0

for qid, docs in qrels_dict.items():
    for doc_id in docs.keys():
        total_gt_docs += 1
        if doc_id in corpus_ids_set:
            matched_gt_docs += 1

print(f"Total ground truth documents: {total_gt_docs}")
print(f"Matched in corpus: {matched_gt_docs}")
print(f"Match rate: {matched_gt_docs/total_gt_docs*100:.2f}%")

if matched_gt_docs == 0:
    print("\nWARNING: No ground truth IDs found in corpus!")
    print("This will cause 0 scores in evaluation.")
    print("\nChecking ID format mismatch...")
    sample_gt_id = list(list(qrels_dict.values())[0].keys())[0]
    sample_corpus_id = str(corpus_df.iloc[0]['doc_id'])
    print(f"Sample GT ID: '{sample_gt_id}' (type: {type(sample_gt_id)})")
    print(f"Sample Corpus ID: '{sample_corpus_id}' (type: {type(sample_corpus_id)})")

=== QRELS SANITY CHECK ===
Number of queries in qrels: 600

First 3 qrels entries:

Query ID: da25227f-1b4e-4907-81e2-a3b3fef62300
  Ground truth docs: {'dc9b290175ed088f3ca2693e3dbe6657': 1}

Query ID: bb8ccee4-864e-44f4-9ea6-598836b4a546
  Ground truth docs: {'86e3f1bdfc14b608cf542abfc4c71dd6': 1}

Query ID: ef6bcdcb-ccd6-430b-9471-8c00c3debf41
  Ground truth docs: {'4ac60b94f1274d11049d9c1890f68841': 1}

=== QA DATAFRAME CHECK ===
QA DataFrame columns: ['qid', 'query', 'retrieval_gt', 'generation_gt']

First query example:
  QID: da25227f-1b4e-4907-81e2-a3b3fef62300
  Query: What are the basic requirements for Alfalfa to grow best, including the ideal soil characteristics, pH range, and the attributes of its root system?
  Retrieval GT: [array(['dc9b290175ed088f3ca2693e3dbe6657'], dtype=object)]
  GT Type: <class 'numpy.ndarray'>

=== CORPUS DOC_ID CHECK ===
First 3 corpus doc_ids:
  dc9b290175ed088f3ca2693e3dbe6657 (type: <class 'str'>)
  86e3f1bdfc14b608cf542abfc4c71dd6 (type: <cl

In [ ]:
# Check for empty contents in corpus_df
empty_contents_count = corpus_df['contents'].isna().sum() + (corpus_df['contents'] == "").sum()
print(f"Total documents: {len(corpus_df)}")
print(f"Documents with empty contents: {empty_contents_count}")

if empty_contents_count > 0:
    print("\nSample empty documents:")
    print(corpus_df[corpus_df['contents'] == ""].head())
else:
    print("No empty documents found in corpus_df.")

Total documents: 2469
Documents with empty contents: 0
No empty documents found in corpus_df.


## 1. Embedding Comparison: Gemini vs OpenAI
We will create two collections in Qdrant, one for each embedding model, and evaluate retrieval performance.


In [ ]:
# Initialize Embedding Models
gemini_embeddings = GoogleGenerativeAIEmbeddings(
    model=Config.GEMINI_MODEL,
    google_api_key=Config.GOOGLE_API_KEY
)

openai_embeddings = OpenAIEmbeddings(
    model=Config.OPENAI_MODEL,
    base_url="https://openrouter.ai/api/v1",
    openai_api_key=Config.OPENROUTER_API_KEY
)

# Initialize Sparse Embedding (SPLADE)
splade_embeddings = FastEmbedSparse(model_name="prithivida/Splade_PP_en_v1")

# Initialize Late Interaction Model (ColBERT) for reranking
colbert_model = LateInteractionTextEmbedding(Config.COLBERT_MODEL)

print("Embedding models initialized.")


Embedding models initialized.


In [ ]:
# def ingest_corpus_unified(collection_name, gemini_model, openai_model, splade_model, colbert_model):
#     """Ingests corpus into a single Qdrant collection with multiple named vectors."""
    
#     # Recreate collection
#     if client.collection_exists(collection_name=collection_name):
#         client.delete_collection(collection_name=collection_name)
    
#     # Define named vectors
#     vectors_config = {
#         "gemini-embedding-001": models.VectorParams(size=3072, distance=models.Distance.COSINE),
#         "text-embedding-3-large": models.VectorParams(size=3072, distance=models.Distance.COSINE),
#         # ColBERT late interaction: multivector, HNSW disabled (not needed for reranking)
#         "jina-colbert-v2": models.VectorParams(
#             size=128,
#             distance=models.Distance.COSINE,
#             multivector_config=models.MultiVectorConfig(
#                 comparator=models.MultiVectorComparator.MAX_SIM
#             ),
#             hnsw_config=models.HnswConfigDiff(m=0)
#         )
#     }
    
#     # Define named sparse vectors
#     sparse_vectors_config = {
#         "splade": models.SparseVectorParams(),
#         "bm25": models.SparseVectorParams(modifier=models.Modifier.IDF)
#     }
        
#     client.create_collection(
#         collection_name=collection_name,
#         vectors_config=vectors_config,
#         sparse_vectors_config=sparse_vectors_config
#     )

#     # Create payload indexes for metadata fields
#     print("Creating payload indexes...")
#     client.create_payload_index(collection_name=collection_name, field_name="metadata.plant", field_schema=models.PayloadSchemaType.TEXT)
#     client.create_payload_index(collection_name=collection_name, field_name="metadata.disease", field_schema=models.PayloadSchemaType.TEXT)
#     client.create_payload_index(collection_name=collection_name, field_name="metadata.type", field_schema=models.PayloadSchemaType.KEYWORD)
#     client.create_payload_index(collection_name=collection_name, field_name="metadata.source", field_schema=models.PayloadSchemaType.KEYWORD)
    
#     points = []
#     batch_size = 25
    
#     print(f"Ingesting into unified collection: {collection_name}...")
#     docs = corpus_df.to_dict('records')
    
#     for i, doc in enumerate(tqdm(docs)):
#         content = doc['contents']
#         doc_id = doc['doc_id']
#         metadata = doc['metadata'] if doc['metadata'] else {}
        
#         # Generate Dense, Sparse & Late Interaction embeddings
#         gemini_vec = gemini_model.embed_query(content)
#         openai_vec = openai_model.embed_query(content)
#         splade_vec = splade_model.embed_query(content)
#         # embed() returns an iterator of 2D arrays (token_count x dim)
#         colbert_vec = list(colbert_model.embed([content]))[0]
        
#         vector_dict = {
#             "gemini-embedding-001": gemini_vec,
#             "text-embedding-3-large": openai_vec,
#             "splade": models.SparseVector(
#                 indices=splade_vec.indices,
#                 values=splade_vec.values
#             ),
#             "bm25": models.Document(
#                 text=content,
#                 model="Qdrant/bm25"
#             ),
#             # ColBERT multivector: list of per-token vectors
#             "jina-colbert-v2": colbert_vec.tolist()
#         }
            
#         point = models.PointStruct(
#             id=doc_id,
#             vector=vector_dict,
#             payload={
#                 "page_content": content,
#                 "metadata": metadata
#             }
#         )
#         points.append(point)
        
#         if len(points) >= batch_size:
#             client.upsert(collection_name=collection_name, points=points)
#             points = []
            
#     if points:
#         client.upsert(collection_name=collection_name, points=points)
        
#     print("Unified ingestion complete.")

# # Run unified ingestion
# ingest_corpus_unified(
#     Config.COLLECTION_NAME, 
#     gemini_embeddings, 
#     openai_embeddings, 
#     splade_embeddings,
#     colbert_model
# )


In [ ]:
client.get_collection(collection_name=Config.COLLECTION_NAME)

CollectionInfo(status=<CollectionStatus.GREEN: 'green'>, optimizer_status=<OptimizersStatusOneOf.OK: 'ok'>, warnings=None, indexed_vectors_count=12335, points_count=2458, segments_count=8, config=CollectionConfig(params=CollectionParams(vectors={'gemini-embedding-001': VectorParams(size=3072, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None, multivector_config=None), 'jina-colbert-v2': VectorParams(size=128, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=HnswConfigDiff(m=0, ef_construct=None, full_scan_threshold=None, max_indexing_threads=None, on_disk=None, payload_m=None, inline_storage=None), quantization_config=None, on_disk=None, datatype=None, multivector_config=MultiVectorConfig(comparator=<MultiVectorComparator.MAX_SIM: 'max_sim'>)), 'text-embedding-3-large': VectorParams(size=3072, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None, multivector_config=None

In [ ]:
def evaluate_retrieval_unified(collection_name, query_text, qid, mode="gemini", k=20):
    """Performs retrieval using named vectors in the unified collection."""
    
    if mode == "gemini":
        vec = gemini_embeddings.embed_query(query_text)
        results = client.query_points(collection_name=collection_name, query=vec, using="gemini-embedding-001", limit=k, with_payload=True)
    elif mode == "openai":
        vec = openai_embeddings.embed_query(query_text)
        results = client.query_points(collection_name=collection_name, query=vec, using="text-embedding-3-large", limit=k, with_payload=True)
    elif mode == "splade":
        sparse_vec = splade_embeddings.embed_query(query_text)
        results = client.query_points(
            collection_name=collection_name,
            query=models.SparseVector(indices=sparse_vec.indices, values=sparse_vec.values),
            using="splade",
            limit=k,
            with_payload=True
        )
    elif mode == "bm25":
        results = client.query_points(
            collection_name=collection_name,
            query=models.Document(text=query_text, model="Qdrant/bm25"),
            using="bm25",
            limit=k,
            with_payload=True
        )
    elif mode == "hybrid_gemini_splade":
        dense_vec = gemini_embeddings.embed_query(query_text)
        sparse_vec = splade_embeddings.embed_query(query_text)
        results = client.query_points(
            collection_name=collection_name,
            prefetch=[
                models.Prefetch(query=dense_vec, using="gemini-embedding-001", limit=k),
                models.Prefetch(
                    query=models.SparseVector(indices=sparse_vec.indices, values=sparse_vec.values),
                    using="splade",
                    limit=k
                ),
            ],
            query=models.FusionQuery(fusion=models.Fusion.RRF),
            limit=k,
            with_payload=True
        )
    elif mode == "hybrid_openai_splade":
        dense_vec = openai_embeddings.embed_query(query_text)
        sparse_vec = splade_embeddings.embed_query(query_text)
        results = client.query_points(
            collection_name=collection_name,
            prefetch=[
                models.Prefetch(query=dense_vec, using="text-embedding-3-large", limit=k),
                models.Prefetch(
                    query=models.SparseVector(indices=sparse_vec.indices, values=sparse_vec.values),
                    using="splade",
                    limit=k
                ),
            ],
            query=models.FusionQuery(fusion=models.Fusion.RRF),
            limit=k,
            with_payload=True
        )
    elif mode == "hybrid_gemini_bm25":
        dense_vec = gemini_embeddings.embed_query(query_text)
        results = client.query_points(
            collection_name=collection_name,
            prefetch=[
                models.Prefetch(query=dense_vec, using="gemini-embedding-001", limit=k),
                models.Prefetch(
                    query=models.Document(text=query_text, model="Qdrant/bm25"),
                    using="bm25",
                    limit=k
                ),
            ],
            query=models.FusionQuery(fusion=models.Fusion.RRF),
            limit=k,
            with_payload=True
        )
    elif mode == "hybrid_openai_bm25":
        dense_vec = openai_embeddings.embed_query(query_text)
        results = client.query_points(
            collection_name=collection_name,
            prefetch=[
                models.Prefetch(query=dense_vec, using="text-embedding-3-large", limit=k),
                models.Prefetch(
                    query=models.Document(text=query_text, model="Qdrant/bm25"),
                    using="bm25",
                    limit=k
                ),
            ],
            query=models.FusionQuery(fusion=models.Fusion.RRF),
            limit=k,
            with_payload=True
        )
    elif mode == "hybrid_gemini_bm25_colbert":
        dense_vec = gemini_embeddings.embed_query(query_text)
        late_vec = list(colbert_model.query_embed([query_text]))[0]
        prefetch_k = max(50, k * 10)
        results = client.query_points(
            collection_name=collection_name,
            prefetch=[
                models.Prefetch(query=dense_vec, using="gemini-embedding-001", limit=prefetch_k),
                models.Prefetch(
                    query=models.Document(text=query_text, model="Qdrant/bm25"),
                    using="bm25",
                    limit=prefetch_k
                ),
            ],
            query=late_vec.tolist(),
            using="jina-colbert-v2",
            limit=k,
            with_payload=True
        )
    elif mode == "hybrid_openai_bm25_colbert":
        dense_vec = openai_embeddings.embed_query(query_text)
        late_vec = list(colbert_model.query_embed([query_text]))[0]
        prefetch_k = max(50, k * 10)
        results = client.query_points(
            collection_name=collection_name,
            prefetch=[
                models.Prefetch(query=dense_vec, using="text-embedding-3-large", limit=prefetch_k),
                models.Prefetch(
                    query=models.Document(text=query_text, model="Qdrant/bm25"),
                    using="bm25",
                    limit=prefetch_k
                ),
            ],
            query=late_vec.tolist(),
            using="jina-colbert-v2",
            limit=k,
            with_payload=True
        )
    
    return results

def evaluate_retrieval_batch_unified(collection_name, queries, mode="gemini", k=20, batch_size=32):
    """Performs batch retrieval using named vectors in the unified collection."""
    all_results = []
    
    for i in tqdm(range(0, len(queries), batch_size), desc=f"Batch Retrieval ({mode})"):
        batch_queries = queries[i : i + batch_size]
        query_texts = [q['query'] for q in batch_queries]
        
        query_requests = []
        
        if mode == "gemini":
            vecs = gemini_embeddings.embed_documents(query_texts)
            for vec in vecs:
                query_requests.append(models.QueryRequest(query=vec, using="gemini-embedding-001", limit=k, with_payload=True))
        elif mode == "openai":
            vecs = openai_embeddings.embed_documents(query_texts)
            for vec in vecs:
                query_requests.append(models.QueryRequest(query=vec, using="text-embedding-3-large", limit=k, with_payload=True))
        elif mode == "splade":
            sparse_vecs = splade_embeddings.embed_documents(query_texts)
            for sparse_vec in sparse_vecs:
                query_requests.append(models.QueryRequest(
                    query=models.SparseVector(indices=sparse_vec.indices, values=sparse_vec.values),
                    using="splade",
                    limit=k,
                    with_payload=True
                ))
        elif mode == "bm25":
            for text in query_texts:
                query_requests.append(models.QueryRequest(
                    query=models.Document(text=text, model="Qdrant/bm25"),
                    using="bm25",
                    limit=k,
                    with_payload=True
                ))
        elif mode.startswith("hybrid") and "colbert" in mode:
            is_gemini = "gemini" in mode
            dense_model = gemini_embeddings if is_gemini else openai_embeddings
            dense_using = "gemini-embedding-001" if is_gemini else "text-embedding-3-large"
            prefetch_k = max(50, k * 10)
            
            dense_vecs = [dense_model.embed_query(text) for text in query_texts]
            late_vecs = list(colbert_model.query_embed(query_texts))
            
            for dv, lv, text in zip(dense_vecs, late_vecs, query_texts):
                query_requests.append(models.QueryRequest(
                    prefetch=[
                        models.Prefetch(query=dv, using=dense_using, limit=prefetch_k),
                        models.Prefetch(
                            query=models.Document(text=text, model="Qdrant/bm25"),
                            using="bm25",
                            limit=prefetch_k
                        ),
                    ],
                    query=lv.tolist(),
                    using="jina-colbert-v2",
                    limit=k,
                    with_payload=True
                ))
        elif mode.startswith("hybrid"):
            is_gemini = "gemini" in mode
            is_splade = "splade" in mode
            
            dense_model = gemini_embeddings if is_gemini else openai_embeddings
            dense_using = "gemini-embedding-001" if is_gemini else "text-embedding-3-large"
            
            dense_vecs = dense_model.embed_documents(query_texts)
            
            if is_splade:
                sparse_vecs = splade_embeddings.embed_documents(query_texts)
                for dv, sv in zip(dense_vecs, sparse_vecs):
                    query_requests.append(models.QueryRequest(
                        prefetch=[
                            models.Prefetch(query=dv, using=dense_using, limit=k),
                            models.Prefetch(
                                query=models.SparseVector(indices=sv.indices, values=sv.values),
                                using="splade",
                                limit=k
                            ),
                        ],
                        query=models.FusionQuery(fusion=models.Fusion.RRF),
                        limit=k,
                        with_payload=True
                    ))
            else:  # bm25
                for dv, text in zip(dense_vecs, query_texts):
                    query_requests.append(models.QueryRequest(
                        prefetch=[
                            models.Prefetch(query=dv, using=dense_using, limit=k),
                            models.Prefetch(
                                query=models.Document(text=text, model="Qdrant/bm25"),
                                using="bm25",
                                limit=k
                            ),
                        ],
                        query=models.FusionQuery(fusion=models.Fusion.RRF),
                        limit=k,
                        with_payload=True
                    ))
        
        batch_res = client.query_batch_points(collection_name=collection_name, requests=query_requests)
        all_results.extend(batch_res)
        
    return all_results


In [ ]:
import pickle
from pathlib import Path

RESULTS_FILE = Path("data/evaluation_results.pkl")

def save_evaluation_state():
    # Convert Run objects to dicts to avoid pickling issues with Numba objects
    serializable_runs = {name: run.to_dict() for name, run in unified_runs.items()}
    state = {
        "unified_results": unified_results,
        "unified_runs": serializable_runs
    }
    # Ensure directory exists
    RESULTS_FILE.parent.mkdir(parents=True, exist_ok=True)
    with open(RESULTS_FILE, "wb") as f:
        pickle.dump(state, f)
    print(f"Evaluation state saved to {RESULTS_FILE}")

def load_evaluation_state():
    if RESULTS_FILE.exists():
        try:
            with open(RESULTS_FILE, "rb") as f:
                state = pickle.load(f)
            print(f"Loaded evaluation state from {RESULTS_FILE}")
            
            # Reconstruct Run objects
            loaded_results = state.get("unified_results", {})
            loaded_runs_dicts = state.get("unified_runs", {})
            loaded_runs = {name: Run.from_dict(d, name=name) for name, d in loaded_runs_dicts.items()}
            
            return loaded_results, loaded_runs
        except Exception as e:
            print(f"Failed to load evaluation state: {e}")
            return {}, {}
    return {}, {}

if 'unified_results' not in globals() or not unified_results:
    unified_results, unified_runs = load_evaluation_state()

# Ensure variables exist even if load failed or file didn't exist
if 'unified_results' not in globals():
    unified_results = {}
if 'unified_runs' not in globals():
    unified_runs = {}

Loaded evaluation state from data\evaluation_results.pkl


In [ ]:
def run_single_evaluation(mode, collection_name=Config.COLLECTION_NAME, k=5, sample_n=None, overwrite=False):
    """Runs evaluation for a single mode. Use sample_n=50 for a quick sanity check."""
    
    # Check if already evaluated and not forcing overwrite
    if not overwrite and not sample_n and mode in unified_results:
        print(f"Mode '{mode}' already evaluated. Loading results...")
        return unified_results[mode]

    # 1. Select Queries (Full or Sample)
    if sample_n:
        # Randomly sample n queries with fixed seed for reproducibility
        queries_df = qa_df.sample(n=min(sample_n, len(qa_df)), random_state=42)
        print(f"⚠️ DEBUG MODE: Running on {len(queries_df)} random queries.")
    else:
        queries_df = qa_df
        print(f"Evaluating mode: {mode} with k={k} on full dataset...")

    queries = queries_df.to_dict('records')
    run_dict = {}
    
    # 2. Run Retrieval (Batched)
    results = evaluate_retrieval_batch_unified(collection_name, queries, mode=mode, k=k)
    
    for q, res in zip(queries, results):
        qid = str(q['qid'])
        run_dict[qid] = {}
        max_rank = k
        for point in res.points:
            str_id = str(point.id).replace("-", "")
            run_dict[qid][str_id] = max_rank
            max_rank -= 1
    
    run = Run(run_dict, name=mode)
    
    # 3. Handle Qrels (Ground Truth)
    # If we only run 50 queries, we must evaluate against ONLY those 50 qrels 
    # otherwise ranx assumes we missed the other 950 queries and gives ~0 score.
    if sample_n:
        sampled_qids = set(run_dict.keys())
        # Filter the global qrels_dict to match our sample
        sampled_qrels_dict = {qid: val for qid, val in qrels_dict.items() if qid in sampled_qids}
        eval_qrels = Qrels(sampled_qrels_dict)
    else:
        eval_qrels = qrels

    metrics = evaluate(eval_qrels, run, metrics=["ndcg@5", "recall@5", "mrr@5"], make_comparable=True)
    
    # 4. Store Results (Only for full runs)
    if not sample_n:
        unified_results[mode] = metrics
        unified_runs[mode] = run
        save_evaluation_state()
    else:
        print(f"⚠️ Results are for SAMPLE only and NOT saved to global stats.")
    
    print(f"Results for {mode}: {metrics}")
    return metrics

## Dense Comparisons

In [ ]:
run_single_evaluation("gemini")
run_single_evaluation("openai")

gemini_results = unified_results.get('gemini')
openai_results = unified_results.get('openai')


Evaluating mode: gemini with k=5 on full dataset...


Batch Retrieval (gemini):   0%|          | 0/19 [00:00<?, ?it/s]

Evaluation state saved to data\evaluation_results.pkl
Results for gemini: {'ndcg@5': 0.9395546696351488, 'recall@5': 0.9741666666666666, 'mrr@5': 0.9365555555555555}
Evaluating mode: openai with k=5 on full dataset...


Batch Retrieval (openai):   0%|          | 0/19 [00:00<?, ?it/s]

Evaluation state saved to data\evaluation_results.pkl
Results for openai: {'ndcg@5': 0.9257698916699928, 'recall@5': 0.9575, 'mrr@5': 0.9324722222222223}


## Hybrid Comparison


In [ ]:
run_single_evaluation("hybrid_gemini_splade")
run_single_evaluation("hybrid_openai_splade")
run_single_evaluation("hybrid_gemini_bm25")
run_single_evaluation("hybrid_openai_bm25")

gemini_hybrid_results = unified_results.get('hybrid_gemini_splade')
openai_hybrid_results = unified_results.get('hybrid_openai_splade')
gemini_bm25_results = unified_results.get('hybrid_gemini_bm25')
openai_bm25_results = unified_results.get('hybrid_openai_bm25')

Evaluating mode: hybrid_gemini_splade with k=5 on full dataset...


Batch Retrieval (hybrid_gemini_splade):   0%|          | 0/19 [00:00<?, ?it/s]

Evaluation state saved to data\evaluation_results.pkl
Results for hybrid_gemini_splade: {'ndcg@5': 0.9307533041186675, 'recall@5': 0.9616666666666667, 'mrr@5': 0.9327777777777779}
Evaluating mode: hybrid_openai_splade with k=5 on full dataset...


Batch Retrieval (hybrid_openai_splade):   0%|          | 0/19 [00:00<?, ?it/s]

Evaluation state saved to data\evaluation_results.pkl
Results for hybrid_openai_splade: {'ndcg@5': 0.9237124925792589, 'recall@5': 0.9566666666666667, 'mrr@5': 0.9299722222222222}
Evaluating mode: hybrid_gemini_bm25 with k=5 on full dataset...


Batch Retrieval (hybrid_gemini_bm25):   0%|          | 0/19 [00:00<?, ?it/s]

Evaluation state saved to data\evaluation_results.pkl
Results for hybrid_gemini_bm25: {'ndcg@5': 0.9414814730926654, 'recall@5': 0.9725, 'mrr@5': 0.9464166666666667}
Evaluating mode: hybrid_openai_bm25 with k=5 on full dataset...


Batch Retrieval (hybrid_openai_bm25):   0%|          | 0/19 [00:00<?, ?it/s]

Evaluation state saved to data\evaluation_results.pkl
Results for hybrid_openai_bm25: {'ndcg@5': 0.9407283256440295, 'recall@5': 0.9666666666666667, 'mrr@5': 0.949861111111111}


## 3. Reranker Comparison: Voyage AI vs ColBERT

- **Voyage AI `rerank-2.5`**: External API reranker applied on top of hybrid retrieval candidates.
- **`jina-colbert-v2`**: Late interaction model (ColBERT) ingested as a multivector in Qdrant and used natively for reranking via `prefetch + query` pattern.


Testing VoyageAIRerank initialization...
Testing reranking...

Reranking Results:
1. The capital of France is Paris. (ID: 2)
2. Apple is a fruit. (ID: 1)


In [ ]:
for key in ["colbert_rerank", "voyageai_rerank2.5"]:
    unified_results.pop(key, None)
    unified_runs.pop(key, None)

save_evaluation_state()

Evaluation state saved to data\evaluation_results.pkl


In [ ]:
# === Sanity Check: Voyage AI Reranker ===
try:
    from langchain_voyageai import VoyageAIRerank
    from langchain_core.documents import Document as LCDocument

    print("Testing VoyageAIRerank initialization...")
    test_reranker = VoyageAIRerank(
        model="rerank-2.5",
        voyageai_api_key=Config.VOYAGE_API_KEY,
        top_k=2
    )

    test_docs = [
        LCDocument(page_content="Apple is a fruit.", metadata={"id": "1"}),
        LCDocument(page_content="The capital of France is Paris.", metadata={"id": "2"}),
        LCDocument(page_content="Python is a programming language.", metadata={"id": "3"})
    ]
    test_query = "What is the capital of France?"

    print("Testing reranking...")
    test_results = test_reranker.compress_documents(test_docs, test_query)
    
    print("\nVoyage AI Reranking Results:")
    for i, doc in enumerate(test_results):
        print(f"  {i+1}. {doc.page_content} (Score: {doc.metadata.get('relevance_score', 'N/A')})")
    print("Voyage AI sanity check PASSED")
except Exception as e:
    print(f"Voyage AI Sanity Check FAILED: {e}")
    import traceback
    traceback.print_exc()

# === Sanity Check: ColBERT (jina-colbert-v2) Late Interaction Reranker via Qdrant ===
try:
    print("\n" + "=" * 60)
    print("Testing ColBERT late interaction reranking via Qdrant...")

    test_query = "What diseases affect tomato plants?"

    # Generate query embeddings
    dense_vec = gemini_embeddings.embed_query(test_query)
    late_vec = list(colbert_model.query_embed([test_query]))[0]

    print(f"Dense vector dim: {len(dense_vec)}")
    print(f"Late interaction vector shape: {late_vec.shape}")

    # Step 1: Hybrid prefetch (dense + BM25)
    prefetch = [
        models.Prefetch(
            query=dense_vec,
            using="gemini-embedding-001",
            limit=20,
        ),
        models.Prefetch(
            query=models.Document(text=test_query, model="Qdrant/bm25"),
            using="bm25",
            limit=20,
        ),
    ]

    # Step 2: Rerank with ColBERT late interaction
    colbert_test_results = client.query_points(
        collection_name=Config.COLLECTION_NAME,
        prefetch=prefetch,
        query=late_vec.tolist(),
        using="jina-colbert-v2",
        with_payload=True,
        limit=5,
    )

    print(f"\nColBERT Reranked Results ({len(colbert_test_results.points)} docs):")
    for i, point in enumerate(colbert_test_results.points):
        content = (point.payload or {}).get('page_content', '')[:100]
        print(f"  {i+1}. [Score: {point.score:.4f}] {content}...")

    # Step 3: Compare with plain hybrid (RRF) to show reranking effect
    hybrid_test_results = client.query_points(
        collection_name=Config.COLLECTION_NAME,
        prefetch=prefetch,
        query=models.FusionQuery(fusion=models.Fusion.RRF),
        limit=5,
        with_payload=True,
    )

    hybrid_ids = [str(p.id) for p in hybrid_test_results.points]
    colbert_ids = [str(p.id) for p in colbert_test_results.points]

    print(f"\nHybrid (RRF) top-5 IDs:      {hybrid_ids}")
    print(f"ColBERT reranked top-5 IDs:  {colbert_ids}")
    reranked_changes = sum(1 for a, b in zip(hybrid_ids, colbert_ids) if a != b)
    print(f"Position changes: {reranked_changes}/5 documents changed position")
    print("ColBERT sanity check PASSED")

except Exception as e:
    print(f"ColBERT Sanity Check FAILED: {e}")
    import traceback
    traceback.print_exc()

Testing VoyageAIRerank initialization...
Testing reranking...

Voyage AI Reranking Results:
  1. The capital of France is Paris. (Score: 0.89453125)
  2. Apple is a fruit. (Score: 0.3046875)
Voyage AI sanity check PASSED

Testing ColBERT late interaction reranking via Qdrant...
Dense vector dim: 3072
Late interaction vector shape: (32, 128)

ColBERT Reranked Results (5 docs):
  1. [Score: 21.3035] Disease: Late Blight Phytophthora Infestans affecting Tomato

Symptoms: Late blight affects all aeri...
  2. [Score: 21.2801] Disease: Bacterial Canker affecting Tomato

Symptoms: Bacterial canker can affect tomato plants of a...
  3. [Score: 21.2251] Disease: Tomato Yellow Leaf Curl Disease Tomato Yellow Leaf Curl Virus (TYLCV) Family Geminiviridae,...
  4. [Score: 21.1289] Disease: Gray Mold (Botrytis Blight) affecting Tomato

Symptoms: Disease appears on tomato seedlings...
  5. [Score: 21.1235] Disease: Tomato Mosaic Virus (ToMV) affecting Tomato

Symptoms: Symptoms can occur at any growt

In [ ]:
# === CONFIGURATION ===
BASE_RETRIEVAL_MODE = "hybrid_gemini_bm25"
CANDIDATE_K = 50

def get_candidates(mode="hybrid_gemini_bm25", collection_name=Config.COLLECTION_NAME, k=100):
    """Retrieves top-k candidates for all queries using the specified retrieval mode (Batched)."""
    candidates = {}
    queries = qa_df.to_dict('records')
    
    print(f"Fetching candidates with mode: {mode} (k={k})...")
    
    results = evaluate_retrieval_batch_unified(collection_name, queries, mode=mode, k=k)
    
    for q, res in zip(queries, results):
        qid = str(q['qid'])
        query_text = q['query']
        
        doc_list = []
        for point in res.points:
            payload = point.payload or {}
            content = payload.get('page_content') or payload.get('contents') or payload.get('text') or ""
            doc_id = str(point.id).replace("-", "")
            doc_list.append((doc_id, content))
            
        candidates[qid] = {
            "query": query_text,
            "docs": doc_list
        }
    return candidates

def evaluate_reranker(reranker_name, candidates=None, overwrite=False):
    key = reranker_name
    
    if not overwrite and key in unified_results:
        print(f"Reranker '{reranker_name}' already evaluated. Loading results...")
        return unified_results[key], unified_runs[key]

    run_dict = {}
    print(f"Reranking with {reranker_name}...")
    
    if reranker_name == "voyageai_rerank2.5":
        from langchain_voyageai import VoyageAIRerank
        from langchain_core.documents import Document as LCDocument
        reranker = VoyageAIRerank(
            model="rerank-2.5",
            voyageai_api_key=Config.VOYAGE_API_KEY,
            top_k=CANDIDATE_K
        )

        for qid, data in tqdm(candidates.items()):
            query = data['query']
            docs = data['docs']
            
            if not docs:
                continue
                
            try:
                valid_docs = [(doc_id, content) for doc_id, content in docs if content and content.strip()]
                if not valid_docs:
                    continue
                
                rerank_docs = [
                    LCDocument(page_content=content, metadata={"doc_id": doc_id})
                    for doc_id, content in valid_docs
                ]
                reranked_results = reranker.compress_documents(rerank_docs, query)
                
                if not reranked_results:
                    continue

                max_rank = len(reranked_results)
                run_dict[qid] = {}
                for res in reranked_results:
                    doc_id = res.metadata["doc_id"]
                    run_dict[qid][doc_id] = max_rank
                    max_rank -= 1
            except Exception as e:
                if len(run_dict) < 5:
                    print(f"Voyage Error for query '{query[:50]}...': {e}")

    elif reranker_name == "colbert_rerank":
        # ColBERT late interaction reranking via Qdrant's native prefetch + query pattern.
        # Prefetch CANDIDATE_K candidates with hybrid search (dense + BM25),
        # then rerank with jina-colbert-v2 multivector using late interaction scoring.
        queries = qa_df.to_dict('records')
        batch_size = 32
        
        for i in tqdm(range(0, len(queries), batch_size), desc="ColBERT Reranking"):
            batch = queries[i:i + batch_size]
            query_texts = [q['query'] for q in batch]
            qids = [str(q['qid']) for q in batch]
            
            dense_vecs = gemini_embeddings.embed_documents(query_texts)
            late_vecs = list(colbert_model.query_embed(query_texts))
            
            query_requests = []
            for dv, lv, text in zip(dense_vecs, late_vecs, query_texts):
                query_requests.append(models.QueryRequest(
                    prefetch=[
                        models.Prefetch(query=dv, using="gemini-embedding-001", limit=CANDIDATE_K),
                        models.Prefetch(
                            query=models.Document(text=text, model="Qdrant/bm25"),
                            using="bm25",
                            limit=CANDIDATE_K
                        ),
                    ],
                    query=lv.tolist(),
                    using="jina-colbert-v2",
                    limit=CANDIDATE_K,
                    with_payload=True
                ))
            
            batch_results = client.query_batch_points(
                collection_name=Config.COLLECTION_NAME,
                requests=query_requests
            )
            
            for qid, res in zip(qids, batch_results):
                run_dict[qid] = {}
                max_rank = len(res.points)
                for point in res.points:
                    doc_id = str(point.id).replace("-", "")
                    run_dict[qid][doc_id] = max_rank
                    max_rank -= 1

    if not run_dict:
        print(f"No results collected for {reranker_name}.")
        return None, None

    run = Run(run_dict, name=f"{reranker_name}_on_{BASE_RETRIEVAL_MODE}")
    metrics = evaluate(qrels, run, metrics=["ndcg@5", "recall@5", "mrr@5"], make_comparable=True)
    
    unified_results[key] = metrics
    unified_runs[key] = run
    save_evaluation_state()
    
    return metrics, run

# --- ColBERT (jina-colbert-v2) Reranker via Qdrant late interaction ---
# Proper: prefetch CANDIDATE_K candidates with hybrid, rerank with ColBERT
colbert_results, colbert_run = evaluate_reranker("colbert_rerank")
print("ColBERT Results:", colbert_results)

# --- Voyage AI Reranker ---
candidates = get_candidates(mode=BASE_RETRIEVAL_MODE, k=CANDIDATE_K)
voyage_results, voyage_run = evaluate_reranker("voyageai_rerank2.5", candidates)
print("Voyage AI Results:", voyage_results)

Reranking with colbert_rerank...


ColBERT Reranking:   0%|          | 0/19 [00:00<?, ?it/s]

Evaluation state saved to data\evaluation_results.pkl
ColBERT Results: {'ndcg@5': 0.9248229700138847, 'recall@5': 0.9508333333333333, 'mrr@5': 0.9364166666666667}
Fetching candidates with mode: hybrid_gemini_bm25 (k=50)...


Batch Retrieval (hybrid_gemini_bm25):   0%|          | 0/19 [00:00<?, ?it/s]

Reranking with voyageai_rerank2.5...


  0%|          | 0/600 [00:00<?, ?it/s]

Evaluation state saved to data\evaluation_results.pkl
Voyage AI Results: {'ndcg@5': 0.9564268366129992, 'recall@5': 0.98, 'mrr@5': 0.9587222222222223}


In [ ]:
unified_results

{'gemini': {'ndcg@5': 0.9395546696351488,
  'recall@5': 0.9741666666666666,
  'mrr@5': 0.9365555555555555},
 'openai': {'ndcg@5': 0.9257698916699928,
  'recall@5': 0.9575,
  'mrr@5': 0.9324722222222223},
 'hybrid_gemini_splade': {'ndcg@5': 0.9307533041186675,
  'recall@5': 0.9616666666666667,
  'mrr@5': 0.9327777777777779},
 'hybrid_openai_splade': {'ndcg@5': 0.9237124925792589,
  'recall@5': 0.9566666666666667,
  'mrr@5': 0.9299722222222222},
 'hybrid_gemini_bm25': {'ndcg@5': 0.9414814730926654,
  'recall@5': 0.9725,
  'mrr@5': 0.9464166666666667},
 'hybrid_openai_bm25': {'ndcg@5': 0.9407283256440295,
  'recall@5': 0.9666666666666667,
  'mrr@5': 0.949861111111111},
 'hybrid_gemini_bm25_colbert': {'ndcg@5': 0.9270903205739832,
  'recall@5': 0.9575,
  'mrr@5': 0.9355833333333333},
 'colbert_rerank': {'ndcg@5': 0.9248229700138847,
  'recall@5': 0.9508333333333333,
  'mrr@5': 0.9364166666666667},
 'voyageai_rerank2.5': {'ndcg@5': 0.9564268366129992,
  'recall@5': 0.98,
  'mrr@5': 0.958722

## Summary of Results


In [ ]:
results_summary = {
    "Gemini (Dense)": unified_results.get("gemini"),
    "OpenAI (Dense)": unified_results.get("openai"),
    "Gemini + SPLADE": unified_results.get("hybrid_gemini_splade"),
    "OpenAI + SPLADE": unified_results.get("hybrid_openai_splade"),
    "Gemini + BM25": unified_results.get("hybrid_gemini_bm25"),
    "OpenAI + BM25": unified_results.get("hybrid_openai_bm25"),
    "Gemini + BM25 + Voyage Rerank 2.5": unified_results.get("voyageai_rerank2.5"),
    "Gemini + BM25 + Jina-colbert-v2": unified_results.get("colbert_rerank"),
}

results_summary = {k: v for k, v in results_summary.items() if v is not None}

df_results = pd.DataFrame(results_summary).T
df_results.sort_values(by="ndcg@5", ascending=False, inplace=True)
df_results


,ndcg@5,recall@5,mrr@5
Gemini + BM25 + Voyage Rerank 2.5,0.956427,0.980000,0.958722
Gemini + BM25,0.941481,0.972500,0.946417
OpenAI + BM25,0.940728,0.966667,0.949861
Gemini (Dense),0.939555,0.974167,0.936556
Gemini + SPLADE,0.930753,0.961667,0.932778
OpenAI (Dense),0.925770,0.957500,0.932472
Gemini + BM25 + Jina-colbert-v2,0.924823,0.950833,0.936417
OpenAI + SPLADE,0.923712,0.956667,0.929972


## Statistical Significance & Visualization
We compare the top performing models to see if the differences are statistically significant and visualize their performance across different recall levels.

In [ ]:
from ranx import compare

# Compare Dense Embeddings
print("=== Statistical Comparison: Gemini vs OpenAI ===")
if 'gemini' in unified_runs and 'openai' in unified_runs:
    report = compare(
        qrels=qrels,
        runs=[unified_runs['gemini'], unified_runs['openai']],
        metrics=["ndcg@5", "mrr@5", "recall@5"],
        max_p=0.05,
        make_comparable=True
    )
    print(report)
else:
    print("Skipping comparison: Missing gemini or openai runs.")

# Compare Best Hybrid vs Rerankers (Voyage vs ColBERT)
print("\n=== Statistical Comparison: Hybrid vs Voyage vs ColBERT Rerankers ===")
rerank_runs = []
for key in ['hybrid_gemini_bm25', 'voyageai_rerank2.5', 'colbert_rerank']:
    if key in unified_runs:
        rerank_runs.append(unified_runs[key])

if len(rerank_runs) > 1:
    report_rerank = compare(
        qrels=qrels,
        runs=rerank_runs,
        metrics=["ndcg@5", "mrr@5", "recall@5"],
        max_p=0.05,
        make_comparable=True
    )
    print(report_rerank)
else:
    print("Skipping comparison: Not enough runs.")


=== Statistical Comparison: Gemini vs OpenAI ===
#    Model    NDCG@5      MRR@5  Recall@5
---  -------  --------  -------  ----------
a    gemini   0.940ᵇ      0.937  0.974ᵇ
b    openai   0.926       0.932  0.958

=== Statistical Comparison: Hybrid vs Voyage vs ColBERT Rerankers ===
#    Model                                     NDCG@5    MRR@5    Recall@5
---  ----------------------------------------  --------  -------  ----------
a    hybrid_gemini_bm25                        0.941ᶜ    0.946    0.973ᶜ
b    voyageai_rerank2.5_on_hybrid_gemini_bm25  0.956ᵃᶜ   0.959ᵃᶜ  0.980ᶜ
c    colbert_rerank_on_hybrid_gemini_bm25      0.925     0.936    0.951


## Qualitative Sample Analysis: Voyage AI vs ColBERT Reranking

Compare top-5 results side-by-side on sample queries, highlighting which docs are relevant (in ground truth).


In [ ]:
import random

N_SAMPLES = 10
SHOW_TOP_K = 5
CONTENT_PREVIEW_LEN = 200

# Build a fast doc_id -> content lookup
corpus_lookup = {
    str(row['doc_id']).replace("-", ""): row['contents']
    for _, row in corpus_df.iterrows()
}

def run_to_plain_dict(run):
    """Convert ranx Run (numba typed dict) to a regular nested Python dict."""
    return {qid: dict(doc_scores) for qid, doc_scores in run.run.items()}

def get_ranked_docs(run_dict, qid, top_k=SHOW_TOP_K):
    """Returns list of (doc_id, score) sorted by score descending."""
    scores = run_dict.get(qid, {})
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_k]

def is_relevant(doc_id, qid):
    return doc_id in qrels_dict.get(qid, {})

# Collect queries where Voyage and ColBERT disagree on top-1
voyage_run = unified_runs.get("voyageai_rerank2.5")
colbert_run = unified_runs.get("colbert_rerank")
baseline_run = unified_runs.get("hybrid_gemini_bm25")

if not (voyage_run and colbert_run):
    print("Missing run data. Run evaluate_reranker() cells first.")
else:
    # Convert numba typed dicts to plain Python dicts once
    voyage_dict = run_to_plain_dict(voyage_run)
    colbert_dict = run_to_plain_dict(colbert_run)
    baseline_dict = run_to_plain_dict(baseline_run) if baseline_run else {}

    common_qids = list(set(voyage_dict.keys()) & set(colbert_dict.keys()))

    # Separate queries into: voyage wins, colbert wins, same top-1
    voyage_win, colbert_win, tied = [], [], []
    for qid in common_qids:
        v_top = get_ranked_docs(voyage_dict, qid, 1)
        c_top = get_ranked_docs(colbert_dict, qid, 1)
        if not v_top or not c_top:
            continue
        v_rel = is_relevant(v_top[0][0], qid)
        c_rel = is_relevant(c_top[0][0], qid)
        if v_rel and not c_rel:
            voyage_win.append(qid)
        elif c_rel and not v_rel:
            colbert_win.append(qid)
        else:
            tied.append(qid)

    print(f"Total common queries: {len(common_qids)}")
    print(f"  Voyage top-1 correct, ColBERT wrong: {len(voyage_win)}")
    print(f"  ColBERT top-1 correct, Voyage wrong: {len(colbert_win)}")
    print(f"  Top-1 same correctness:               {len(tied)}")

    # Sample: mix of voyage_win, colbert_win, and tied
    random.seed(42)
    sample_qids = []
    sample_qids += random.sample(voyage_win, min(4, len(voyage_win)))
    sample_qids += random.sample(colbert_win, min(4, len(colbert_win)))
    sample_qids += random.sample(tied, min(2, len(tied)))
    random.shuffle(sample_qids)
    sample_qids = sample_qids[:N_SAMPLES]

    # Build comparison DataFrame rows
    rows = []
    for qid in sample_qids:
        query_text = qa_df[qa_df['qid'].astype(str) == qid]['query'].values
        query_text = query_text[0] if len(query_text) > 0 else "(unknown)"

        v_docs = get_ranked_docs(voyage_dict, qid)
        c_docs = get_ranked_docs(colbert_dict, qid)
        b_docs = get_ranked_docs(baseline_dict, qid)

        for rank_idx in range(SHOW_TOP_K):
            v_id, v_score = v_docs[rank_idx] if rank_idx < len(v_docs) else (None, None)
            c_id, c_score = c_docs[rank_idx] if rank_idx < len(c_docs) else (None, None)
            b_id, b_score = b_docs[rank_idx] if rank_idx < len(b_docs) else (None, None)

            rows.append({
                "qid": qid,
                "query": query_text[:80] + "..." if len(query_text) > 80 else query_text,
                "rank": rank_idx + 1,
                "baseline_doc": (corpus_lookup.get(b_id, "")[:CONTENT_PREVIEW_LEN] if b_id else ""),
                "baseline_rel": "YES" if b_id and is_relevant(b_id, qid) else "no",
                "voyage_doc": (corpus_lookup.get(v_id, "")[:CONTENT_PREVIEW_LEN] if v_id else ""),
                "voyage_rel": "YES" if v_id and is_relevant(v_id, qid) else "no",
                "colbert_doc": (corpus_lookup.get(c_id, "")[:CONTENT_PREVIEW_LEN] if c_id else ""),
                "colbert_rel": "YES" if c_id and is_relevant(c_id, qid) else "no",
            })

    df_sample = pd.DataFrame(rows)

    # Display per query
    for qid in sample_qids:
        sub = df_sample[df_sample['qid'] == qid]
        query_text = sub.iloc[0]['query']
        gt_ids = set(qrels_dict.get(qid, {}).keys())
        gt_previews = [corpus_lookup.get(gid, "")[:120] for gid in list(gt_ids)[:2]]

        print("=" * 100)
        print(f"QID: {qid}")
        print(f"Query: {query_text}")
        print(f"Ground Truth ({len(gt_ids)} doc(s)): {gt_previews[0][:120] if gt_previews else '?'}...")
        print()
        print(f"{'Rank':<5} {'Baseline':^42} {'Voyage AI':^42} {'ColBERT':^42}")
        print(f"{'':5} {'[Rel] Preview':<42} {'[Rel] Preview':<42} {'[Rel] Preview':<42}")
        print("-" * 130)
        for _, r in sub.iterrows():
            print(
                f"#{int(r['rank']):<4} "
                f"[{r['baseline_rel']:<3}] {r['baseline_doc'][:36]:<36}  "
                f"[{r['voyage_rel']:<3}] {r['voyage_doc'][:36]:<36}  "
                f"[{r['colbert_rel']:<3}] {r['colbert_doc'][:36]}"
            )
        print()


Total common queries: 600
  Voyage top-1 correct, ColBERT wrong: 24
  ColBERT top-1 correct, Voyage wrong: 8
  Top-1 same correctness:               568
QID: bc61220d-cbb5-4af7-9c8c-1075a53c97d4
Query: What is the method of apple tree propagation that involves joining the lower par...
Ground Truth (1 doc(s)): Cultivation and propagation of Apple (Part 1)

Apple trees grow best in the tropics and at higher latitudes. They requir...

Rank                   Baseline                                  Voyage AI                                   ColBERT                  
      [Rel] Preview                              [Rel] Preview                              [Rel] Preview                             
----------------------------------------------------------------------------------------------------------------------------------
#1    [YES] Cultivation and propagation of Apple  [YES] Cultivation and propagation of Apple  [no ] Cultivation and propagation of Manda
#2    [no ] Cultivation an